# M1 — CNN + LSTM Baseline (CV-STQA)

**Architecture:** Video → 16 uniformly sampled frames → ResNet-50 → frame features → LSTM → 6 quality scores

This follows the manual's M1 definition: `I_t → ResNet-50 → F_t → LSTM → Q_O`. The manual specifies **T = 16** with uniform temporal sampling for the initial implementation. fileciteturn4file0

Outputs are saved so M2 and later models can be compared against the same M1 test predictions and metrics.

In [1]:
# CELL 1 — Install / imports

!pip -q install scipy scikit-learn opencv-python-headless

import os
import json
import glob
import math
import time
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from scipy.stats import spearmanr, kendalltau, pearsonr
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [2]:
# CELL 2 — Locate the input files automatically

INPUT_ROOT = Path("/kaggle/input")

def find_file(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {INPUT_ROOT}")
    return matches[0]

TRAIN_CSV = find_file("train.csv")
VAL_CSV   = find_file("val.csv")
TEST_CSV  = find_file("test.csv")

# Optional M0 config
m0_matches = list(INPUT_ROOT.rglob("M0_config.json"))
M0_CONFIG_PATH = m0_matches[0] if m0_matches else None

print("train.csv:", TRAIN_CSV)
print("val.csv  :", VAL_CSV)
print("test.csv :", TEST_CSV)
print("M0 config:", M0_CONFIG_PATH)


train.csv: /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/train.csv
val.csv  : /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/val.csv
test.csv : /kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/test.csv
M0 config: /kaggle/input/datasets/miruthulajmm/m0-config-json/M0_config.json


In [3]:
# CELL 3 — Load splits and inspect columns

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())
print("\nFirst rows:")
display(train_df.head())


Train: (700, 8)
Val  : (150, 8)
Test : (150, 8)

Columns:
['video_id', 'TSV', 'B', 'SR', 'S', 'U', 'O', 'video_path']

First rows:


,video_id,TSV,B,SR,S,U,O,video_path
0,68-11.mp4,2.0025,2.1235,1.8600,1.72120,1.8126,3.5140,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...
1,48-3.mp4,2.1210,3.0000,1.6898,2.30670,2.3584,4.1275,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...
2,2-10.mp4,1.4000,1.6566,1.3648,0.68874,1.4057,2.9560,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...
3,40-1.mp4,1.8593,0.8656,0.4560,1.27030,1.0326,2.2650,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...
4,52-15.mp4,1.6436,1.7560,0.8960,1.98970,1.6381,3.3420,/kaggle/input/datasets/miruthulajmm/mac-vqa-vi...


In [4]:
# CELL 4 — Identify video-ID and the six target columns

# MAC-VQA quality dimensions from the manual:
# TSV, B, SR, S, U, O
TARGETS = ["TSV", "B", "SR", "S", "U", "O"]

# Handle common naming variations automatically.
def resolve_column(df, candidates):
    lower = {c.lower().strip(): c for c in df.columns}
    for x in candidates:
        if x.lower() in lower:
            return lower[x.lower()]
    return None

target_cols = {}
aliases = {
    "TSV": ["TSV", "T_SV", "temporal_spatial_visibility", "temporal-spatial visibility"],
    "B":   ["B", "brightness"],
    "SR":  ["SR", "specular_reflection", "specular reflection"],
    "S":   ["S", "stability"],
    "U":   ["U", "utility", "utility_cleanliness", "utility/cleanliness"],
    "O":   ["O", "overall", "overall_quality", "overall quality"],
}

for t in TARGETS:
    col = resolve_column(train_df, aliases[t])
    if col is not None:
        target_cols[t] = col

print("Detected target columns:", target_cols)

if len(target_cols) != 6:
    print("\nCould not automatically identify all six targets.")
    print("Available columns:", train_df.columns.tolist())
    raise ValueError(
        "Please edit the aliases in CELL 4 so they match your CSV column names."
    )

# Find the video identifier column.
ID_CANDIDATES = [
    "video", "video_id", "video_name", "filename", "file_name",
    "clip", "clip_id", "id", "name", "path"
]

id_col = resolve_column(train_df, ID_CANDIDATES)

if id_col is None:
    # Pick the first non-target object/string column.
    non_targets = [c for c in train_df.columns if c not in target_cols.values()]
    string_cols = [c for c in non_targets if train_df[c].dtype == "object"]
    if string_cols:
        id_col = string_cols[0]

if id_col is None:
    raise ValueError("Could not identify the video ID/name column.")

print("Video ID column:", id_col)
print("Targets:", [target_cols[t] for t in TARGETS])


Detected target columns: {'TSV': 'TSV', 'B': 'B', 'SR': 'SR', 'S': 'S', 'U': 'U', 'O': 'O'}
Video ID column: video_id
Targets: ['TSV', 'B', 'SR', 'S', 'U', 'O']


In [6]:
# CELL 5 — Locate the video dataset

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".mpg", ".mpeg"}

video_files = []
for ext in VIDEO_EXTS:
    video_files.extend(INPUT_ROOT.rglob(f"*{ext}"))

video_files = sorted(set(video_files))

print("Number of video files found:", len(video_files))
print("Example videos:")
for p in video_files[:10]:
    print(p)

if len(video_files) == 0:
    raise FileNotFoundError(
        "No video files found under /kaggle/input. Add your video Kaggle dataset as an Input."
    )


Number of video files found: 1000
Example videos:
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-11.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-12.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-13.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-14.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-15.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-16.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-17.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-18.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-19.mp4
/kaggle/input/datasets/miruthulajmm/mac-vqa-videos/videos/1-22.mp4


In [7]:
# CELL 6 — Match CSV IDs to actual video files

# Build lookup using several possible filename forms.
video_lookup = {}

for p in video_files:
    name = p.name
    stem = p.stem
    rel = str(p)

    keys = {
        name.lower(),
        stem.lower(),
        rel.lower(),
    }

    for k in keys:
        video_lookup[k] = p

def find_video(video_id):
    s = str(video_id).strip()
    candidates = [
        s.lower(),
        Path(s).name.lower(),
        Path(s).stem.lower(),
    ]

    for c in candidates:
        if c in video_lookup:
            return video_lookup[c]

    # Last-resort stem comparison.
    stem = Path(s).stem.lower()
    matches = [p for p in video_files if p.stem.lower() == stem]
    if matches:
        return matches[0]

    return None

for df in [train_df, val_df, test_df]:
    df["video_path"] = df[id_col].apply(find_video)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = df["video_path"].isna().sum()
    print(f"{name}: {len(df)} rows | missing videos: {missing}")
    if missing:
        print(df.loc[df["video_path"].isna(), id_col].head(10).tolist())

if train_df["video_path"].isna().any() or val_df["video_path"].isna().any() or test_df["video_path"].isna().any():
    raise FileNotFoundError(
        "Some CSV IDs could not be matched to videos. Check the video filenames and ID column."
    )


train: 700 rows | missing videos: 0
val: 150 rows | missing videos: 0
test: 150 rows | missing videos: 0


In [8]:
# CELL 7 — Configuration

NUM_FRAMES = 16          # Manual's initial temporal sampling choice
IMAGE_SIZE = 224         # ResNet-50 input
FEATURE_DIM = 2048       # ResNet-50 penultimate feature size

LSTM_HIDDEN = 256
LSTM_LAYERS = 1
DROPOUT = 0.30

BATCH_SIZE = 8
NUM_EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5

# Frozen ResNet-50 keeps M1 fast and isolates the effect of temporal modeling.
FREEZE_RESNET = True

OUTPUT_DIR = Path("/kaggle/working/m1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("T =", NUM_FRAMES)


Output directory: /kaggle/working/m1_outputs
T = 16


In [9]:
# CELL 8 — Uniform frame sampling + ResNet preprocessing

def sample_frame_indices(num_video_frames, T=NUM_FRAMES):
    if num_video_frames <= 0:
        raise ValueError("Video has no readable frames.")

    if num_video_frames == 1:
        return np.zeros(T, dtype=np.int64)

    return np.linspace(0, num_video_frames - 1, T).round().astype(np.int64)

resnet_weights = models.ResNet50_Weights.DEFAULT
frame_transform = resnet_weights.transforms()

print("Using pretrained:", resnet_weights)


Using pretrained: ResNet50_Weights.IMAGENET1K_V2


In [10]:
# CELL 9 — ResNet-50 feature extractor

resnet = models.resnet50(weights=resnet_weights)
resnet.fc = nn.Identity()
resnet = resnet.to(DEVICE)
resnet.eval()

if FREEZE_RESNET:
    for p in resnet.parameters():
        p.requires_grad = False

print("ResNet-50 loaded.")
print("Feature dimension:", FEATURE_DIM)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 254MB/s]


ResNet-50 loaded.
Feature dimension: 2048


In [11]:
# CELL 10 — Extract 16-frame ResNet features

@torch.no_grad()
def extract_video_features(video_path):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()
        raise RuntimeError(f"Could not read frames from {video_path}")

    indices = sample_frame_indices(total)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()

        if not ok:
            # Fall back to the last successfully read frame.
            if frames:
                frames.append(frames[-1])
                continue
            cap.release()
            raise RuntimeError(f"Failed to read frame {idx} from {video_path}")

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()

    batch = torch.stack([
        frame_transform(torch.from_numpy(frame).permute(2, 0, 1))
        for frame in frames
    ]).to(DEVICE)

    feats = resnet(batch)
    return feats.cpu().numpy().astype(np.float32)


def build_feature_cache(df, split_name):
    cache_dir = OUTPUT_DIR / "frame_features" / split_name
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_paths = []

    for row_idx, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Extracting {split_name} features"
    ):
        out = cache_dir / f"{row_idx}.npy"

        if not out.exists():
            feats = extract_video_features(row["video_path"])
            np.save(out, feats)

        feature_paths.append(str(out))

    return feature_paths


In [12]:
# CELL 11 — Precompute frame-level features

# This is the expensive CNN step.
# It is done once because the ResNet backbone is frozen.
train_df["feature_path"] = build_feature_cache(train_df, "train")
val_df["feature_path"]   = build_feature_cache(val_df, "val")
test_df["feature_path"]  = build_feature_cache(test_df, "test")

print("Feature extraction complete.")
print("Each file should have shape:", (NUM_FRAMES, FEATURE_DIM))
print("Example:", np.load(train_df.iloc[0]["feature_path"]).shape)


Extracting train features:   0%|          | 0/700 [00:00<?, ?it/s]

Extracting val features:   0%|          | 0/150 [00:00<?, ?it/s]

Extracting test features:   0%|          | 0/150 [00:00<?, ?it/s]

Feature extraction complete.
Each file should have shape: (16, 2048)
Example: (16, 2048)


In [13]:
# CELL 12 — Dataset for LSTM training

class FeatureSequenceDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        x = np.load(row["feature_path"]).astype(np.float32)
        y = np.array(
            [float(row[target_cols[t]]) for t in TARGETS],
            dtype=np.float32
        )

        return torch.from_numpy(x), torch.from_numpy(y)


train_ds = FeatureSequenceDataset(train_df)
val_ds   = FeatureSequenceDataset(val_df)
test_ds  = FeatureSequenceDataset(test_df)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))


Train batches: 88
Val batches: 19
Test batches: 19


In [14]:
# CELL 13 — M1 CNN + LSTM model

class M1CNNLSTM(nn.Module):
    def __init__(
        self,
        feature_dim=FEATURE_DIM,
        hidden_dim=LSTM_HIDDEN,
        num_layers=LSTM_LAYERS,
        dropout=DROPOUT,
        num_outputs=6
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False
        )

        self.dropout = nn.Dropout(dropout)
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_outputs)
        )

    def forward(self, x):
        # x: [B, T, 2048]
        seq_out, (h_n, c_n) = self.lstm(x)

        # Last temporal state represents the sequence.
        last = h_n[-1]       # [B, hidden]
        out = self.regressor(self.dropout(last))

        return out


model = M1CNNLSTM().to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(model)
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")


M1CNNLSTM(
  (lstm): LSTM(2048, 256, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (regressor): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=6, bias=True)
  )
)
Trainable parameters: 2,395,014
Total parameters: 2,395,014


In [15]:
# CELL 14 — Training setup

criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def run_epoch(loader, training=True):
    model.train(training)

    total_loss = 0.0
    total_n = 0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            pred = model(x)
            loss = criterion(pred, y)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n += bs

    return total_loss / total_n


In [16]:
# CELL 15 — Train M1

best_val_loss = float("inf")
best_state = None
bad_epochs = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    train_loss = run_epoch(train_loader, training=True)
    val_loss = run_epoch(val_loader, training=False)

    scheduler.step(val_loss)

    lr_now = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": lr_now
    })

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train={train_loss:.5f} | "
        f"val={val_loss:.5f} | "
        f"lr={lr_now:.2e} | "
        f"time={time.time()-start:.1f}s"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        bad_epochs = 0
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        torch.save(
            best_state,
            OUTPUT_DIR / "m1_lstm_best.pt"
        )
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "m1_training_history.csv", index=False)

print("Best validation loss:", best_val_loss)


Epoch 01/20 | train=0.89200 | val=0.30507 | lr=1.00e-03 | time=1.7s
Epoch 02/20 | train=0.37713 | val=0.23682 | lr=1.00e-03 | time=0.8s
Epoch 03/20 | train=0.33519 | val=0.20661 | lr=1.00e-03 | time=0.8s
Epoch 04/20 | train=0.26236 | val=0.20476 | lr=1.00e-03 | time=0.8s
Epoch 05/20 | train=0.23761 | val=0.19546 | lr=1.00e-03 | time=0.8s
Epoch 06/20 | train=0.24568 | val=0.19862 | lr=1.00e-03 | time=0.8s
Epoch 07/20 | train=0.21554 | val=0.22227 | lr=1.00e-03 | time=0.8s
Epoch 08/20 | train=0.20969 | val=0.18727 | lr=1.00e-03 | time=0.8s
Epoch 09/20 | train=0.18342 | val=0.18446 | lr=1.00e-03 | time=0.8s
Epoch 10/20 | train=0.18211 | val=0.17952 | lr=1.00e-03 | time=0.8s
Epoch 11/20 | train=0.19435 | val=0.25221 | lr=1.00e-03 | time=0.9s
Epoch 12/20 | train=0.18493 | val=0.18606 | lr=1.00e-03 | time=0.8s
Epoch 13/20 | train=0.17379 | val=0.17451 | lr=1.00e-03 | time=0.8s
Epoch 14/20 | train=0.15844 | val=0.16850 | lr=1.00e-03 | time=0.8s
Epoch 15/20 | train=0.15864 | val=0.16162 | lr=1

In [17]:
# CELL 16 — Load best model and generate predictions

model.load_state_dict(torch.load(
    OUTPUT_DIR / "m1_lstm_best.pt",
    map_location="cpu"
))
model = model.to(DEVICE)
model.eval()

@torch.no_grad()
def predict_loader(loader):
    preds = []
    targets = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        pred = model(x).cpu().numpy()

        preds.append(pred)
        targets.append(y.numpy())

    return np.concatenate(preds), np.concatenate(targets)


val_pred, val_true = predict_loader(val_loader)
test_pred, test_true = predict_loader(test_loader)

print("Val predictions:", val_pred.shape)
print("Test predictions:", test_pred.shape)


Val predictions: (150, 6)
Test predictions: (150, 6)


In [18]:
# CELL 17 — Calculate SRCC, PLCC, KRCC, RMSE

def safe_spearman(y_true, y_pred):
    return spearmanr(y_true, y_pred).statistic

def safe_pearson(y_true, y_pred):
    return pearsonr(y_true, y_pred).statistic

def safe_kendall(y_true, y_pred):
    return kendalltau(y_true, y_pred).statistic

def calculate_metrics(y_true, y_pred):
    rows = []

    for i, target in enumerate(TARGETS):
        yt = y_true[:, i]
        yp = y_pred[:, i]

        rows.append({
            "dimension": target,
            "SRCC": safe_spearman(yt, yp),
            "PLCC": safe_pearson(yt, yp),
            "KRCC": safe_kendall(yt, yp),
            "RMSE": np.sqrt(mean_squared_error(yt, yp))
        })

    return pd.DataFrame(rows)


val_metrics = calculate_metrics(val_true, val_pred)
test_metrics = calculate_metrics(test_true, test_pred)

print("M1 Validation Metrics")
display(val_metrics)

print("M1 Test Metrics")
display(test_metrics)

val_metrics.to_csv(OUTPUT_DIR / "m1_val_metrics.csv", index=False)
test_metrics.to_csv(OUTPUT_DIR / "m1_test_metrics.csv", index=False)


M1 Validation Metrics


,dimension,SRCC,PLCC,KRCC,RMSE
0,TSV,0.868204,0.891167,0.683183,0.344567
1,B,0.696040,0.742644,0.513116,0.523749
2,SR,0.758637,0.778466,0.565597,0.425148
3,S,0.911050,0.924568,0.740850,0.320591
4,U,0.881320,0.886296,0.701107,0.349344
5,O,0.914732,0.923527,0.745343,0.413509


M1 Test Metrics


,dimension,SRCC,PLCC,KRCC,RMSE
0,TSV,0.867574,0.880608,0.683728,0.345008
1,B,0.687808,0.732273,0.509913,0.548093
2,SR,0.792789,0.801638,0.595865,0.398585
3,S,0.890768,0.903171,0.717047,0.338146
4,U,0.869849,0.868754,0.681625,0.356199
5,O,0.902981,0.908550,0.727411,0.446560


In [19]:
# CELL 18 — Save test predictions

test_predictions = test_df[[id_col]].copy()

for i, target in enumerate(TARGETS):
    test_predictions[f"{target}_true"] = test_true[:, i]
    test_predictions[f"{target}_pred"] = test_pred[:, i]

test_predictions.to_csv(
    OUTPUT_DIR / "m1_test_predictions.csv",
    index=False
)

display(test_predictions.head())


,video_id,TSV_true,TSV_pred,B_true,B_pred,SR_true,SR_pred,S_true,S_pred,U_true,U_pred,O_true,O_pred
0,24-14.mp4,1.8909,2.002073,0.8560,0.877567,2.7880,1.998101,2.05030,1.957324,1.7000,1.910390,3.4743,3.511255
1,59-4.mp4,2.5554,2.279038,2.4565,1.829449,2.2695,2.498692,2.38500,2.250770,2.4293,2.372166,4.5701,4.134795
2,8-14.mp4,1.5745,0.736986,0.8990,0.846529,0.4520,0.815201,0.83866,0.415553,0.6560,0.700670,1.7056,1.445353
3,11-11.mp4,1.6540,1.756256,1.4376,1.775747,1.4656,1.670806,0.90522,1.507846,1.8565,1.867821,2.7560,3.227115
4,48-4.mp4,2.7590,2.205735,2.1233,1.463271,1.7565,1.604230,2.36900,2.089114,2.0300,2.012296,4.1635,3.643572


In [20]:
# CELL 19 — Save M1 configuration

m1_config = {
    "model": "M1_CNN_LSTM",
    "architecture": "16 frames -> ResNet-50 -> 2048-D frame features -> LSTM -> 6 regression outputs",
    "num_frames": NUM_FRAMES,
    "sampling": "uniform",
    "image_size": IMAGE_SIZE,
    "resnet": "ResNet-50 pretrained",
    "resnet_feature_dim": FEATURE_DIM,
    "resnet_frozen": FREEZE_RESNET,
    "lstm_hidden": LSTM_HIDDEN,
    "lstm_layers": LSTM_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "epochs_max": NUM_EPOCHS,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "early_stopping_patience": PATIENCE,
    "targets": TARGETS,
    "target_columns": target_cols,
    "seed": SEED,
    "train_csv": str(TRAIN_CSV),
    "val_csv": str(VAL_CSV),
    "test_csv": str(TEST_CSV)
}

with open(OUTPUT_DIR / "M1_config.json", "w") as f:
    json.dump(m1_config, f, indent=2)

print(json.dumps(m1_config, indent=2))


{
  "model": "M1_CNN_LSTM",
  "architecture": "16 frames -> ResNet-50 -> 2048-D frame features -> LSTM -> 6 regression outputs",
  "num_frames": 16,
  "sampling": "uniform",
  "image_size": 224,
  "resnet": "ResNet-50 pretrained",
  "resnet_feature_dim": 2048,
  "resnet_frozen": true,
  "lstm_hidden": 256,
  "lstm_layers": 1,
  "dropout": 0.3,
  "batch_size": 8,
  "epochs_max": 20,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "early_stopping_patience": 5,
  "targets": [
    "TSV",
    "B",
    "SR",
    "S",
    "U",
    "O"
  ],
  "target_columns": {
    "TSV": "TSV",
    "B": "B",
    "SR": "SR",
    "S": "S",
    "U": "U",
    "O": "O"
  },
  "seed": 42,
  "train_csv": "/kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/train.csv",
  "val_csv": "/kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/val.csv",
  "test_csv": "/kaggle/input/datasets/miruthulajmm/cv-stqa-configuration-split/test.csv"
}


In [21]:
# CELL 20 — Final output checklist

print("M1 COMPLETE")
print("=" * 60)

for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(OUTPUT_DIR))


M1 COMPLETE
M1_config.json
frame_features/test/0.npy
frame_features/test/1.npy
frame_features/test/10.npy
frame_features/test/100.npy
frame_features/test/101.npy
frame_features/test/102.npy
frame_features/test/103.npy
frame_features/test/104.npy
frame_features/test/105.npy
frame_features/test/106.npy
frame_features/test/107.npy
frame_features/test/108.npy
frame_features/test/109.npy
frame_features/test/11.npy
frame_features/test/110.npy
frame_features/test/111.npy
frame_features/test/112.npy
frame_features/test/113.npy
frame_features/test/114.npy
frame_features/test/115.npy
frame_features/test/116.npy
frame_features/test/117.npy
frame_features/test/118.npy
frame_features/test/119.npy
frame_features/test/12.npy
frame_features/test/120.npy
frame_features/test/121.npy
frame_features/test/122.npy
frame_features/test/123.npy
frame_features/test/124.npy
frame_features/test/125.npy
frame_features/test/126.npy
frame_features/test/127.npy
frame_features/test/128.npy
frame_features/test/129.npy
